In [5]:
import os
import re
import pandas as pd
from datetime import datetime

SOURCE_CSV_CHECKOUT = r"833789_2026-07-11_14_45_47.csv"
SOURCE_CSV_CHECKIN  = r"322037_2026-07-11_14_46_52.csv"
OUTPUT_CSV          = r"c:\Users\cmara\OneDrive\Documents\python\py_practice\RC_guesty_gsheets\guesty_res.csv"

EXCLUDE_PROPERTIES = {
    # Austin, TX
    "1117 Brookswood A", "1117 Brookswood B",
    "1126 Brookswood A", "1126 Brookswood B",
    "1229 Delano A",     "1229 Delano B",
    "1231 Delano A",     "1231 Delano B",
    "4807 Prock A",      "4807 Prock B&C",
    "6108 Club terrace",
    "707 Valdez",
    # New Orleans, LA / Bay St. Louis, MS
    "307 Main",
    "315 Main 1A", "315 Main 1B", "315 Main 1C", "315 Main 1E",
    "315 Main 2F",  "315 Main 2H", "315 Main 2I", "315 Main 2J",
    "1401 Delachaise A", "1401 Delachaise B", "1401 Delachaise C", "1401 Delachaise D",
    "4417 Dryades",
    "468 St Joseph",
    "1560 Magazine",
    "1562 Magazine 1A", "1562 Magazine 1B", "1562 Magazine 2A", "1562 Magazine 2B",
    # Savannah, GA
    "105 Duffy 1", "105 Duffy 2",
    "107 W Park",
    "1312 Abercorn",
    "220 W Park",
    "302 W Park",
    "311 W York 1", "311 W York 2",
    "315 Duffy",
    "319 Congress",
    "320 W Bolton",
    "417 E Bay",
    "440 Habersham",
    "524 E Jones",
    "710 Bernard",
}
_EXCLUDE_LOWER = {p.lower() for p in EXCLUDE_PROPERTIES}

df_checkout = pd.read_csv(SOURCE_CSV_CHECKOUT)
df_checkin  = pd.read_csv(SOURCE_CSV_CHECKIN)

df_checkout.columns = df_checkout.columns.str.strip()
df_checkin.columns  = df_checkin.columns.str.strip()

print("df_checkout columns:", df_checkout.columns.tolist())
print("df_checkin columns :", df_checkin.columns.tolist())
print(f"Rows — checkout: {len(df_checkout)}, checkin: {len(df_checkin)}")
print()

def normalize_property(listing: str) -> list:
    raw = listing.split('/')[0].strip()
    raw = re.sub(r'^F2X?\s+', '', raw)

    abbreviations = {
        'Grav':       'Gravier',
        'Gra':        'Gravier',
        'Baronn':     'Baronne',
        'Baro':       'Baronne',
        'Caron':      'Carondelet',
        'Barth':      'Bartholomew',
        'Brookswo':   'Brookswood',
        'Broo':       'Brookswood',
        'Webbervill': 'Webberville',
        'Webber':     'Webberville',
        'Montgom':    'Montgomery',
        'Mont':       'Montgomery',
        'Dela':       'Delano',
        'Con':        'Congress',
        'S Ramp':     'S Rampart',
        'OCH':        'Oretha Castle Haley',
        'Tchoupit':   'Tchoupitoulas',
        'MLK':        'Martin Luther King',
    }
    for abbr, full in abbreviations.items():
        raw = re.sub(rf'\b{abbr}\b', full, raw)

    # Remove all version/unit markers (V, V1, V2, VI, VII, U) wherever they appear
    raw = re.sub(r'\s*\b(V(?:I+|\d*)|U)\b', '', raw).strip()
    # Strip any trailing & left behind by combined-unit listing names
    raw = raw.rstrip('&').strip()

    # "704-715 & N 2nd": hyphen-range street numbers + & + street name
    match = re.match(r'^(\d+)-(\d+)\s*&\s*(.+)$', raw)
    if match:
        num1, num2, street = match.group(1), match.group(2), match.group(3).strip()
        return [f"{num1} {street}", f"{num2} {street}"]

    # "1308&12 Baronne": two street numbers where second may be a short suffix
    match = re.match(r'^(\d+)&(\d+)\s+(.+)$', raw)
    if match:
        num1, num2, street = match.group(1), match.group(2), match.group(3)
        if len(num2) < len(num1):
            num2 = num1[:len(num1) - len(num2)] + num2
        return [f"{num1} {street}", f"{num2} {street}"]

    # "422 Gravier 201&202": digit unit list after street name
    match = re.match(r'^(.*?\d+\s+\w+)\s+([\d]+(?:&[\d]+)+)$', raw)
    if match:
        base    = match.group(1)
        units   = match.group(2).split('&')
        ref_len = len(units[0])
        return [f"{base} {u.zfill(ref_len)}" for u in units]

    # "1229 Dela A&B": letter unit list after address
    match = re.match(r'^(.+)\s+([A-Z](?:&[A-Z])+)$', raw)
    if match:
        base  = match.group(1)
        units = match.group(2).split('&')
        return [f"{base} {u}" for u in units]

    return [raw]

_DT_FORMATS = (
    "%Y-%m-%d %I:%M %p",
    "%m/%d/%Y %H:%M",
)

def parse_dt(value: str) -> datetime:
    for fmt in _DT_FORMATS:
        try:
            return datetime.strptime(value, fmt)
        except ValueError:
            continue
    raise ValueError(f"Unrecognised datetime format: {value!r}")

_CHECKOUT_STD = datetime.strptime("11:00 AM", "%I:%M %p").time()
_CHECKIN_STD  = datetime.strptime("04:00 PM", "%I:%M %p").time()

def compute_adjustments(co_time: str, ci_time: str) -> str:
    codes = []
    if co_time:
        t = datetime.strptime(co_time, "%I:%M %p").time()
        if t < _CHECKOUT_STD:
            codes.append("ECO")
        elif t > _CHECKOUT_STD:
            codes.append("LCO")
    if ci_time:
        t = datetime.strptime(ci_time, "%I:%M %p").time()
        if t < _CHECKIN_STD:
            codes.append("ECI")
        elif t > _CHECKIN_STD:
            codes.append("LCI")
    return ", ".join(codes)

# Seed city lookup from existing output, then overwrite with live source CSV data
city_lookup = {}
if os.path.exists(OUTPUT_CSV):
    try:
        df_existing = pd.read_csv(OUTPUT_CSV)
        if 'City' in df_existing.columns and 'Property' in df_existing.columns:
            for _, row in df_existing.iterrows():
                city = str(row['City']).strip()
                if city and city.lower() != 'nan':
                    city_lookup[str(row['Property']).strip()] = city
        print(f"City lookup seeded from existing output: {len(city_lookup)} entries")
    except Exception as e:
        print(f"Could not read existing output for city lookup: {e}")
else:
    print("No existing output file — City column will be empty on first run")
print()

checkout_lookup = {}
checkin_lookup  = {}
guest_lookup    = {}   # checkin guest takes priority over checkout guest for T/O rows

# Checkout CSV: check-out times + departing guest + city
for row in df_checkout.to_dict(orient="records"):
    props   = normalize_property(row['LISTING'])
    co_dt   = parse_dt(f"{row['CHECK-OUT DATE']} {row['CHECK-OUT TIME']}")
    co_date = co_dt.strftime("%Y-%m-%d")
    city    = str(row.get("LISTING'S CITY", '')).strip()
    for prop in props:
        checkout_lookup[(prop, co_date)] = co_dt.strftime("%I:%M %p")
        guest_lookup[(prop, co_date)]    = (row.get('CONFIRMATION CODE', ''), row.get('GUEST', ''))
        if city and city.lower() != 'nan':
            city_lookup[prop] = city

# Checkin CSV: check-in times + arriving guest (overwrites checkout guest for T/O) + city
for row in df_checkin.to_dict(orient="records"):
    props   = normalize_property(row['LISTING'])
    ci_dt   = parse_dt(f"{row['CHECK-IN DATE']} {row['CHECK-IN TIME']}")
    ci_date = ci_dt.strftime("%Y-%m-%d")
    city    = str(row.get("LISTING'S CITY", '')).strip()
    for prop in props:
        checkin_lookup[(prop, ci_date)] = ci_dt.strftime("%I:%M %p")
        guest_lookup[(prop, ci_date)]   = (row.get('CONFIRMATION CODE', ''), row.get('GUEST', ''))
        if city and city.lower() != 'nan':
            city_lookup[prop] = city

all_events = set(checkout_lookup.keys()) | set(checkin_lookup.keys())

output_rows = []
for (prop, date) in all_events:
    if prop.lower() in _EXCLUDE_LOWER:
        continue
    date_obj  = datetime.strptime(date, "%Y-%m-%d")
    co_time   = checkout_lookup.get((prop, date), "")
    ci_time   = checkin_lookup.get((prop, date), "")
    conf_code, guest = guest_lookup.get((prop, date), ('', ''))
    output_rows.append({
        'City':              city_lookup.get(prop, ""),
        'Day':               date_obj.strftime("%A"),
        'Date':              date,
        'Confirmation Code': conf_code,
        'Guest':             guest,
        'Property':          prop,
        'Check-out Time':    co_time,
        'Check-in Time':     ci_time,
        'T/O':               "yes" if (co_time and ci_time) else "",
        'Adjustments':       compute_adjustments(co_time, ci_time),
    })

df_out = pd.DataFrame(output_rows, columns=[
    'City', 'Day', 'Date', 'Confirmation Code', 'Guest', 'Property',
    'Check-out Time', 'Check-in Time', 'T/O', 'Adjustments'
])
df_out = df_out.sort_values(by=['Date', 'City', 'Property']).reset_index(drop=True)

print(df_out.head(20).to_string(index=False))
print()
print("--- Sample T/O rows ---")
print(df_out[df_out['T/O'] == "yes"].head(10).to_string(index=False))
print()
print(f"Total rows: {len(df_out)}  |  Turnovers: {(df_out['T/O'] == 'yes').sum()}")

df_out.to_csv(OUTPUT_CSV, index=False)
print(f"\nExported to: {OUTPUT_CSV}")


df_checkout columns: ['CONFIRMATION CODE', 'LISTING', 'GUEST', 'CHECK-OUT DATE', 'CHECK-OUT TIME']
df_checkin columns : ['CONFIRMATION CODE', 'LISTING', 'GUEST', 'CHECK-IN DATE', 'CHECK-IN TIME']
Rows — checkout: 413, checkin: 404

City lookup seeded from existing output: 162 entries

       City      Day       Date Confirmation Code              Guest                   Property Check-out Time Check-in Time T/O Adjustments
            Saturday 2026-07-11        HM2QP5Z8X9       Steven Stair            520 E Harris CH                     04:00 PM                
     Austin Saturday 2026-07-11        HMTHPJPCA9    Kristine Capili    1802 Martin Luther King                     04:00 PM                
New Orleans Saturday 2026-07-11        HMDA3NXP88       Amy Sprinkle            1213 Magazine 1                     04:00 PM                
New Orleans Saturday 2026-07-11        HM58KYRXNT    Aimee Broussard               1324 Baronne                     03:00 PM             ECI
New Orlea